In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/project1/setup_catalogs/utilities

In [0]:
dbutils.widgets.text('catalog', 'project1_cat', 'catalog')
dbutils.widgets.text('data_source', 'customers', 'data_source')

In [0]:
catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')

In [0]:
print(catalog, data_source)

In [0]:
base_path = f's3://sportsbar-childpro/{data_source}/*.csv'
print(base_path)

In [0]:
df = (spark.read.format('csv')\
            .option('header', True)\
            .option('inferSchema', True)\
            .load(base_path)\
            .withColumn('read_timestamp', F.current_timestamp())
            )
display(df.limit(10))

In [0]:
df.write\
.format('delta')\
.option('delta.enableChangeDataFeed', 'true')\
.mode('overwrite')\
.saveAsTable(f'{catalog}.{bronze_schema}.{data_source}')

### Silver **processing**

In [0]:
df_bronze = spark.sql(f'''
SELECT * FROM {catalog}.{bronze_schema}.{data_source}''')
display(df_bronze.limit(10))

In [0]:
df_bronze.printSchema()

In [0]:
df_duplicates = df_bronze.groupby('customer_id').count().filter('count > 1')
display(df_duplicates)

In [0]:
df_silver = df_bronze.dropDuplicates(['customer_id'])
display(df_silver)

In [0]:
df_duplicates = df_silver.groupby('customer_id').count().filter('count > 1')
display(df_duplicates)

In [0]:
display(df_silver.filter(F.col('customer_name') != F.trim(F.col('customer_name'))))


In [0]:
df_silver = df_silver.withColumn('customer_name', F.trim(F.col('customer_name')))

In [0]:
display(df_silver)

In [0]:
display(df_silver.filter(F.col('customer_name') != F.trim(F.col('customer_name'))))

In [0]:
df_silver.select('city').distinct().show()

In [0]:
city_mapping = {
    'Bengalore' : 'Bengaluru',
    'Bengaluruu' : 'Bengaluru',

    'Hyderabadd' : 'Hyderabad',
    'Hyderbad' : 'Hyderabad',

    'NewDelhee' : 'New Delhi',
    'NewDelhi' : 'New Delhi',
    'NewDheli' : 'New Delhi'
}

allowed_city = ['Bengaluru', 'Hyderabad', 'New Delhi']
df_silver = df_silver.replace(city_mapping, subset=['city'])\
                    .withColumn('city', F.when(F.col('city').isNull(), None)\
                    .when(F.col('city').isin(allowed_city), F.col('city'))\
                    .otherwise(None))

In [0]:
df_silver.select('city').distinct().show()

In [0]:
df_silver.select('customer_name').distinct().show()

In [0]:
df_silver = df_silver.withColumn('customer_name', F.initcap(F.col('customer_name')))

In [0]:
df_silver.select('customer_name').distinct().show()

In [0]:
df_silver.filter(F.col('city').isNull()).show(truncate=False) 

In [0]:
null_customer_name = ['Sprintx Nutrition', 'Zenathlete Foods', 'Primefuel Nutrition', 'Recovery Lane']

In [0]:
df_silver.filter(F.col('customer_name').isin(null_customer_name)).show(truncate=False)

In [0]:
# City corrections Confirmation by Business team

customer_city_fix = {

    789403 : 'New Delhi',

    789420 : 'Bengaluru',

    789421 : 'Hyderabad',

    789521 : 'Hyderabad',

    789603 : 'Hyderabad'
}

df_fix = spark.createDataFrame(
    [(k,v) for k,v in customer_city_fix.items()],
    ['customer_id', 'fixed_city']
)

display(df_fix)

In [0]:
df_silver = (df_silver.join(df_fix, "customer_id", "left").withColumn('city', F.coalesce('city', 'fixed_city')).drop('fixed_city'))

In [0]:
display(df_silver.limit(10))

In [0]:
df_silver.filter(F.col('customer_name').isin(null_customer_name)).show(truncate=False)

In [0]:
df_silver = df_silver.withColumn('customer_id', F.col('customer_id').cast('string'))

In [0]:
df_silver.printSchema()

In [0]:
display(df_silver)

In [0]:
df_silver = (df_silver.withColumn(
        "customer",
         F.concat_ws("-", "customer_name", F.coalesce(F.col("city"), F.lit("Unknown"))))
         .withColumn("market", F.lit("India"))
         .withColumn("platform", F.lit("Sports Bar"))
         .withColumn("channel", F.lit("Acquisition"))
)

In [0]:
display(df_silver)

In [0]:
df_silver.write.format('delta').option('delta.enableChangeDataFeed','true').option('mergeSchema','true').mode("overwrite").saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

### Gold Processing

In [0]:
df_silver = spark.sql(f'select * from {catalog}.{silver_schema}.{data_source}')

In [0]:
df_gold = df_silver.select('customer_id', 'customer', 'customer_name', 'city', 'market', 'platform', 'channel')

display(df_gold)

In [0]:
df_gold.write.format('delta').option('delta.enableChangeDataFeed','true').mode("overwrite").saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")

In [0]:
display(df_gold)

In [0]:
delta_table = DeltaTable.forName(spark, 'project1_cat.gold.dim_customers')

df_child_customers = spark.table('project1_cat.gold.sb_dim_customers').select(
    F.col('customer_id').alias('customer_code'),
    'customer',
    'market',
    'platform',
    'channel'
)

In [0]:
delta_table.alias('target').merge(
    source=df_child_customers.alias('source'),
    condition=F.expr('target.customer_code = source.customer_code')
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
%sql
select * from project1_cat.gold.dim_customers